# Proyecto Fashion Peaks: 02 - Entrenamiento de CNN para Regresión Continua
Este notebook implementa el enfoque de **Regresión Continua de Área** sugerido en la actualización del 13 de mayo por el profesor Edgardo Bonzi.

### ¿Por qué Regresión?
En el enfoque por clasificación, el profesor detectó que en el rango de áreas entre 9250 y 17250 la red colapsaba clasificando sistemáticamente en la clase 16250.
Al tratar el área como una variable física continua (número real entre 0 y 20.000):
1. La red penaliza proporcionalmente la distancia del error (no trata un error cercano igual que uno lejano).
2. Se eliminan los artefactos de discretización de 500 unidades.
3. Se obtiene una estimación directa del área del pico de Cesio-137.


In [ ]:
# 1. Configuración de entorno y conexión con Google Drive
import sys, os

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    PATH_DRIVE = '/content/gdrive/MyDrive/Fashion_Peaks_Colab'
    if os.path.exists(PATH_DRIVE):
        os.chdir(PATH_DRIVE)
    print(f"-> Conectado a Google Colab. Directorio actual: {os.getcwd()}")
else:
    print(f"-> Entorno local. Directorio actual: {os.getcwd()}")


In [ ]:
# 2. Importaciones necesarias
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# 3. Dataset de Regresión con Normalización
ESCALA_MAX_AREA = 20000.0

class SpectraRegressionDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = self.labels_df.iloc[idx, 0]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert('L')
        area_real = float(self.labels_df.iloc[idx, 1])
        area_norm = area_real / ESCALA_MAX_AREA
        if self.transform:
            image = self.transform(image)
        label = torch.tensor([area_norm], dtype=torch.float32)
        return image, label, area_real


In [ ]:
# 4. Copia al disco local SSD de Colab y partición Train / Val / Test
os.system("cp -r imagenes_entrenamiento_regresion /content/ 2>/dev/null || true")

CSV_FILE = "imagenes_entrenamiento_regresion/labels.csv"
ROOT_DIR = "/content/imagenes_entrenamiento_regresion" if os.path.exists("/content/imagenes_entrenamiento_regresion") else "imagenes_entrenamiento_regresion"

transform = transforms.Compose([
    transforms.ToTensor()
])

full_dataset = SpectraRegressionDataset(csv_file=CSV_FILE, root_dir=ROOT_DIR, transform=transform)

total_samples = len(full_dataset)
train_size = int(0.70 * total_samples)
val_size   = int(0.15 * total_samples)
test_size  = total_samples - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)

print(f"Total imágenes para Regresión: {total_samples}")
print(f"  - Entrenamiento: {len(train_dataset)}")
print(f"  - Validación:    {len(val_dataset)}")
print(f"  - Test:          {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# 5. Arquitectura de la CNN para Regresión (SpectraCNNRegressor)
class SpectraCNNRegressor(nn.Module):
    def __init__(self):
        super(SpectraCNNRegressor, self).__init__()
        
        # Convoluciones idénticas a la red del profesor
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.AdaptiveAvgPool2d((1, 1))
        
        # Capas densas adaptadas a una salida escalar continua
        self.fc1 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 1)  # 1 salida: Área continua

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

model = SpectraCNNRegressor().to(device)


In [ ]:
# 6. Bucle de Entrenamiento y Validación
EPOCHS = 60
# HuberLoss es ideal: combina MSE para errores pequeños y L1 para errores grandes
criterion = nn.HuberLoss(delta=0.05)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

os.makedirs("modelos_guardados", exist_ok=True)
BEST_MODEL_PATH = "modelos_guardados/modelo_regresion_mejorado.pth"

train_losses, val_losses = [], []
train_maes, val_maes = [], []
best_val_loss = float('inf')

print(f"Iniciando entrenamiento de Regresión por {EPOCHS} épocas...")
for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    running_loss, total_err_train, count_train = 0.0, 0.0, 0
    for images, labels_norm, areas_reales in train_loader:
        images, labels_norm = images.to(device), labels_norm.to(device)
        optimizer.zero_grad()
        preds_norm = model(images)
        loss = criterion(preds_norm, labels_norm)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        # Desnormalizar para calcular error en unidades de área
        preds_reales = preds_norm.detach().cpu().numpy().flatten() * ESCALA_MAX_AREA
        reales = areas_reales.numpy().flatten()
        total_err_train += np.sum(np.abs(preds_reales - reales))
        count_train += len(reales)

    epoch_train_loss = running_loss / count_train
    epoch_train_mae  = total_err_train / count_train

    # --- Validation ---
    model.eval()
    running_val_loss, total_err_val, count_val = 0.0, 0.0, 0
    with torch.no_grad():
        for images, labels_norm, areas_reales in val_loader:
            images, labels_norm = images.to(device), labels_norm.to(device)
            preds_norm = model(images)
            loss = criterion(preds_norm, labels_norm)
            running_val_loss += loss.item() * images.size(0)
            
            preds_reales = preds_norm.cpu().numpy().flatten() * ESCALA_MAX_AREA
            reales = areas_reales.numpy().flatten()
            total_err_val += np.sum(np.abs(preds_reales - reales))
            count_val += len(reales)

    epoch_val_loss = running_val_loss / count_val
    epoch_val_mae  = total_err_val / count_val
    
    scheduler.step(epoch_val_loss)
    
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    train_maes.append(epoch_train_mae)
    val_maes.append(epoch_val_mae)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        guardado = " [* Guardado]"
    else:
        guardado = ""

    if (epoch + 1) % 5 == 0 or epoch == 0 or guardado:
        print(f"Época [{epoch+1:02d}/{EPOCHS:02d}] "
              f"Loss: {epoch_train_loss:.5f} (MAE: {epoch_train_mae:.1f}) | "
              f"Val Loss: {epoch_val_loss:.5f} (MAE: {epoch_val_mae:.1f}){guardado}")

print()
print(f"Mejor modelo de regresión guardado en: {BEST_MODEL_PATH}")


In [ ]:
# 7. Curvas de Aprendizaje (Pérdida y Error Absoluto Medio)
epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label='Train Loss')
plt.plot(epochs_range, val_losses, label='Val Loss')
plt.title('Huber Loss por Época')
plt.xlabel('Época')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_maes, label='Train MAE')
plt.plot(epochs_range, val_maes, label='Val MAE')
plt.title('Error Absoluto Medio (MAE) en Unidades de Área')
plt.xlabel('Época')
plt.ylabel('Unidades de Área')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# 8. Evaluación en el Conjunto de Test y Gráfico de Dispersión
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

y_test_reales, y_test_preds = [], []

with torch.no_grad():
    for images, labels_norm, areas_reales in test_loader:
        images = images.to(device)
        preds_norm = model(images)
        preds_reales = preds_norm.cpu().numpy().flatten() * ESCALA_MAX_AREA
        
        y_test_reales.extend(areas_reales.numpy().flatten())
        y_test_preds.extend(preds_reales)

y_test_reales = np.array(y_test_reales)
y_test_preds  = np.array(y_test_preds)

mae_test  = mean_absolute_error(y_test_reales, y_test_preds)
rmse_test = np.sqrt(mean_squared_error(y_test_reales, y_test_preds))
r2_test   = r2_score(y_test_reales, y_test_preds)

print("=== MÉTRICAS EN CONJUNTO DE PRUEBA (TEST SET) ===")
print(f"MAE  (Error Absoluto Medio) : {mae_test:.2f} unidades de área")
print(f"RMSE (Raíz Error Cuadrático): {rmse_test:.2f} unidades de área")
print(f"R²   (Coef. Determinación)  : {r2_test:.4f}")

# Gráfico de Dispersión: Real vs Predicho
plt.figure(figsize=(7, 7))
plt.scatter(y_test_reales, y_test_preds, alpha=0.6, color='blue', edgecolors='k', s=30, label='Muestras de Test')
plt.plot([0, ESCALA_MAX_AREA], [0, ESCALA_MAX_AREA], 'r--', linewidth=2, label='Predicción Ideal (y = x)')
plt.xlim(0, ESCALA_MAX_AREA)
plt.ylim(0, ESCALA_MAX_AREA)
plt.xlabel("Área Real")
plt.ylabel("Área Predicha")
plt.title("Regresión: Área Real vs. Predicha por la CNN")
plt.legend(); plt.grid(True)
plt.show()


In [ ]:
# 9. Análisis Específico de la Región Crítica (9250 a 17250)
# El profesor notó que en clasificación esta zona colapsaba en la clase 16250.
# Evaluamos cómo se comporta la regresión aquí:
mask_critica = (y_test_reales >= 9250) & (y_test_reales <= 17250)
muestras_criticas = np.sum(mask_critica)

if muestras_criticas > 0:
    mae_critico = mean_absolute_error(y_test_reales[mask_critica], y_test_preds[mask_critica])
    r2_critico = r2_score(y_test_reales[mask_critica], y_test_preds[mask_critica])
    print(f"--- Desempeño en Zona Crítica [9250 - 17250] ({muestras_criticas} muestras) ---")
    print(f"MAE en la región conflictiva: {mae_critico:.2f} unidades")
    print(f"R² en la región conflictiva : {r2_critico:.4f}")
    
    # Histograma de errores residuales en esta zona
    residuos = y_test_preds[mask_critica] - y_test_reales[mask_critica]
    plt.figure(figsize=(7, 4))
    plt.hist(residuos, bins=15, color='teal', edgecolor='black')
    plt.axvline(0, color='red', linestyle='--')
    plt.xlabel("Error Residual (Predicho - Real)")
    plt.ylabel("Frecuencia")
    plt.title("Distribución de Residuos en el Rango [9250 - 17250]")
    plt.grid(True)
    plt.show()
else:
    print("No cayeron muestras en esa ventana en este split de test.")
